# OpenAI Audio Analysis Pipeline (Test)

Minimal, reliable pipeline using OpenAI API for transcription and diarization.
No PyTorch, local Whisper models, or pyannote - API-only implementation.

In [ ]:
# Setup: Import libraries and load environment variables
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables (API key)
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define paths
AUDIO_FILE = "../data/US_DebateAudio.wav"
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ Environment loaded")
print(f"✓ Audio file: {AUDIO_FILE}")
print(f"✓ Output directory: {OUTPUT_DIR}")

In [ ]:
# OpenAI Diarized Transcription
# Request transcription with speaker labels and timestamps

print("Sending audio to OpenAI for transcription with diarization...")

# Open audio file and send to OpenAI
with open(AUDIO_FILE, "rb") as audio_file:
    response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="verbose_json",
        timestamp_granularities=["segment"]
    )

# Convert response to dict and save
response_dict = response.model_dump()
output_json = OUTPUT_DIR / "openai_diarized.json"
with open(output_json, "w") as f:
    json.dump(response_dict, f, indent=2)

print(f"✓ Transcription complete")
print(f"✓ Language: {response_dict.get('language', 'N/A')}")
print(f"✓ Duration: {response_dict.get('duration', 'N/A'):.2f}s")
print(f"✓ Segments: {len(response_dict.get('segments', []))}")
print(f"✓ Saved to: {output_json}")

In [ ]:
# Speaker Transcript Table
# Convert diarized segments into structured DataFrame

# Extract segments from response
segments = response_dict.get('segments', [])

# Build DataFrame with speaker info
# Note: OpenAI Whisper API doesn't provide native speaker diarization
# We'll use a simple heuristic: assign speakers based on pauses
data = []
current_speaker = "Speaker_1"
speaker_count = 1

for i, seg in enumerate(segments):
    # Simple speaker change detection: if pause > 2 seconds, assume new speaker
    if i > 0:
        prev_end = segments[i-1]['end']
        curr_start = seg['start']
        pause_duration = curr_start - prev_end
        
        if pause_duration > 2.0:  # 2 second threshold for speaker change
            speaker_count += 1
            current_speaker = f"Speaker_{speaker_count}"
    
    data.append({
        'speaker': current_speaker,
        'start': seg['start'],
        'end': seg['end'],
        'duration': seg['end'] - seg['start'],
        'text': seg['text'].strip()
    })

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
output_csv = OUTPUT_DIR / "speaker_transcript.csv"
df.to_csv(output_csv, index=False)

print(f"✓ Created transcript table with {len(df)} segments")
print(f"✓ Detected {speaker_count} speakers")
print(f"✓ Saved to: {output_csv}")
print("\nFirst few rows:")
print(df.head())

In [ ]:
# Basic Analytics
# Calculate speaking time, turns, and words per minute per speaker

# Total speaking time per speaker
speaking_time = df.groupby('speaker')['duration'].sum()

# Number of turns per speaker
turns = df.groupby('speaker').size()

# Words per minute per speaker
def count_words(text):
    return len(text.split())

df['word_count'] = df['text'].apply(count_words)
total_words = df.groupby('speaker')['word_count'].sum()
words_per_minute = (total_words / speaking_time) * 60

# Create analytics summary
analytics = pd.DataFrame({
    'Speaker': speaking_time.index,
    'Total Speaking Time (s)': speaking_time.values,
    'Number of Turns': turns.values,
    'Total Words': total_words.values,
    'Words Per Minute': words_per_minute.values
})

print("Speaker Analytics Summary:")
print("="*70)
print(analytics.to_string(index=False))
print("="*70)

# Save analytics
output_analytics = OUTPUT_DIR / "speaker_analytics.csv"
analytics.to_csv(output_analytics, index=False)
print(f"\n✓ Saved analytics to: {output_analytics}")

In [ ]:
# Visualization: Speaking Time Distribution
# Pie chart showing proportion of speaking time per speaker

plt.figure(figsize=(10, 7))

# Create pie chart
colors = plt.cm.Set3(range(len(speaking_time)))
plt.pie(speaking_time.values, 
        labels=speaking_time.index, 
        autopct='%1.1f%%',
        colors=colors,
        startangle=90)

plt.title('Speaking Time Distribution by Speaker', fontsize=14, fontweight='bold')
plt.axis('equal')

# Save visualization
output_viz = OUTPUT_DIR / "speaking_time_distribution.png"
plt.savefig(output_viz, dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved visualization to: {output_viz}")